In [1]:
import rebayes_mini
import jax
jax.config.update("jax_enable_x64", True)
import chex
import numpy as np
import pandas as pd
import seaborn as sns
import jax.numpy as jnp
import matplotlib.pyplot as plt
from functools import partial
from rebayes_mini import callbacks
from rebayes_mini.states import GaussState
import pickle, os, sys, time, platform
from tqdm import tqdm

from profilter import ExtendedKalmanFilter
from profilter_rebuttal import PrOFilter
from unscented import UnscentedKalmanFilter, UKFPrOFilterPred, CubatureKalmanFilter, CKFPrOFilterPred, UKFPrOFilterTaylor

cmap = {
    "EKF": "lightseagreen",
    "UKF": "teal",
    "CKF": "slateblue",
    "EKF-PrO": "mediumorchid",
    "UKF-PrO": "deeppink",
    "CKF-PrO": "darkgoldenrod",
}

def euler_step(y, i, dt, f):
    h = dt
    t = dt * i
    k1 = h * f(y, t, i)
    return y + k1

def rk4_step(y, i, dt, f):
    h = dt
    t = dt * i
    k1 = h * f(y, t, i)
    k2 = h * f(y + k1 / 2, dt * i + h / 2, i)
    k3 = h * f(y + k2 / 2, t + h / 2, i)
    k4 = h * f(y + k3, t + h, i)

    y_next = y + 1 / 6 * (k1 + 2 * k2 + 2 * k3 + k4)
    return y_next

@partial(jax.jit, static_argnames=("f",))
def solve(ys, dt, N, f):
    @jax.jit
    def step(i, ys):
        ysi = euler_step(ys[i - 1], i, dt, f)
        return ys.at[i].set(ysi)
    return jax.lax.fori_loop(1, N, step, ys)

In [2]:
D = 60
N = 1000
dt = 0.002
sigma = 1.0
F = 8.0
n_hidden = 2
transition_prob = 0.995
observation_covariance = 5.0

@partial(jax.vmap, in_axes=(None, 0, None))
def fcoord(x, k, D):
    xdot = (x[(k + 1) % D] - x[k - 2]) * x[k - 1] - x[k] + F
    return xdot

key = jax.random.PRNGKey(315)
key_init, key_sim, key_eval = jax.random.split(key, 3)
key_state, key_measurement = jax.random.split(key_sim)

key_hidden, key_state = jax.random.split(key_state)
transition_state = jnp.ones((n_hidden, n_hidden)) * 1/(n_hidden-1) * (1-transition_prob)
transition_state = transition_state.at[jnp.diag_indices_from(transition_state)].set(transition_prob)

key_hidden, key_state = jax.random.split(key_state)
phi_hidden = jax.random.normal(key_hidden, (D, n_hidden-1)) * sigma
phi_hidden = jnp.concatenate([jnp.zeros((D, 1)), phi_hidden], -1)

@jax.jit
def f(x, t, i, regime):
    keyt = jax.random.fold_in(key_state, i)
    key_regime, keyt = jax.random.split(keyt)
    probabilities = transition_state[regime, :]
    logits = jnp.log(probabilities)
    regime_next = jax.random.categorical(key_regime, logits)
    ixs = jnp.arange(D)
    xdot = fcoord(x, ixs, D) + phi_hidden[:, regime_next]
    return xdot, regime_next

x0 = jax.random.normal(key_init, (D,))
x_init = jnp.copy(x0)
xs = jnp.zeros((N,) + x0.shape)
xs = xs.at[0].set(x0)
regime = int(0)
for i in range(1, N):
    xdot, regime = f(x0, i * dt, i, regime)
    x0 = x0 + dt * xdot
    xs = xs.at[i].set(x0)

ys = xs + jnp.sqrt(observation_covariance) * jax.random.normal(key_measurement, xs.shape)

def latent_fn(x):
    @jax.jit
    def f(x, t, _):
        ixs = jnp.arange(D)
        return fcoord(x, ixs, D)
    return euler_step(x, 0, dt, f)

def obs_fn(x, _):
    return x

def get_mean_and_cov(bel, bel_pred, y, x):
    return bel.mean, bel.cov

def nll_step(x_true, mean, cov):
    d = mean.shape[0]
    diff = x_true - mean
    sign, logdet = jnp.linalg.slogdet(cov)
    quad = diff @ jnp.linalg.solve(cov, diff)
    return 0.5 * (quad + logdet + d * jnp.log(2 * jnp.pi))

In [3]:
method = "UKF-PrO-Taylor"
nsteps = xs.shape[0]

t0 = time.time()
agent_taylor = UKFPrOFilterTaylor(
        latent_fn, obs_fn,
        dynamics_covariance=jnp.eye(D) * jnp.square(sigma * dt),
        observation_covariance=observation_covariance * jnp.eye(D),
        n_inner=5, n_outer=10,
    )
init_bel = agent_taylor.init_bel(x_init, cov=1.0)
filterfn = partial(agent_taylor.scan, callback_fn=get_mean_and_cov)
_, (hist_taylor, hist_taylor_cov) = filterfn(init_bel, ys, jnp.ones(nsteps))

mse_taylor = jnp.sum(jnp.square(hist_taylor - xs))
nll_taylor = jax.vmap(nll_step)(xs, hist_taylor, hist_taylor_cov)

time_taylor = time.time() - t0

print(f"{method}: MSE = {mse_taylor:.4f}, mean NLL = {jnp.mean(nll_taylor):.4f}, time = {time_taylor:.1f}s")

UKF-PrO-Taylor: MSE = 7813.8538, mean NLL = 544.1627, time = 8.7s


In [5]:
method = "UKF-PrO-Taylor"
nsteps = xs.shape[0]

t0 = time.time()
agent_taylor = UKFPrOFilterTaylor(
        latent_fn, obs_fn,
        dynamics_covariance=jnp.eye(D) * jnp.square(sigma * dt),
        observation_covariance=observation_covariance * jnp.eye(D),
        n_inner=5, n_outer=10,
    )
init_bel = agent_taylor.init_bel(x_init, cov=1.0)
filterfn = partial(agent_taylor.scan, callback_fn=get_mean_and_cov)
_, (hist_taylor, hist_taylor_cov) = filterfn(init_bel, ys, jnp.ones(nsteps))

mse_taylor = jnp.sum(jnp.square(hist_taylor - xs))
nll_taylor = jax.vmap(nll_step)(xs, hist_taylor, hist_taylor_cov)

time_taylor = time.time() - t0


In [6]:
print(f"{method}: MSE = {mse_taylor:.4f}, mean NLL = {jnp.mean(nll_taylor):.4f}, time = {time_taylor:.1f}s")

UKF-PrO-Taylor: MSE = 7813.8538, mean NLL = 544.1627, time = 6.3s
